# Inference using LSDB-style API

In [1]:
import lsdb
import hyrax

In [2]:
from distributed import Client, LocalCluster

N_WORKERS, THREADS_PER_WORKER = 1, 1
cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 1,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54796,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:54801,Total threads: 1
Dashboard: http://127.0.0.1:54802/status,Memory: 32.00 GiB
Nanny: tcp://127.0.0.1:54799,


## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [3]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [4]:
def create_meta_dict(nested_frame):
    meta_dict = {}
    cols = nested_frame._operation.meta.columns
    nested_cols = nested_frame._operation.meta.nested_columns

    non_nested_cols = [col for col in cols if col not in nested_cols]

    for col in non_nested_cols:
        meta_dict[col] = str(nested_frame[col].dtype)

    for col in nested_cols:
        meta_dict[col] = {}
        for ncols in nested_frame[col].columns:
            meta_dict[col][ncols] = str(nested_frame[f"{col}.{ncols}"].dtype)

    return meta_dict

In [5]:
h = hyrax.Hyrax()
h.config["model"]["name"] = "HyraxLoopback"
h.config["data_loader"]["batch_size"] = 64

h.config["data_request"] = {
    "infer": {
        "data": {
            "dataset_class": "DataframeDataset",
            "data_location": create_meta_dict(tess_filtered_nans),
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj", "lightcurve_sap_flux"],
        }
    }
}

h.set_config("data_set.use_cache", False)
h.set_config("infer.model_weights_file", "none")

## Use a Dask Worker Plugin to provide a model for use with inference

In [ ]:
import uuid

import torch
import numpy as np
import pandas as pd
from distributed import WorkerPlugin, get_worker

from hyrax.models.model_registry import fetch_model_class
from hyrax.models.model_utils import load_model_weights
from hyrax.plugin_utils import load_prepare_inputs
from hyrax.pytorch_ignite import _auto_model, create_process_func, dist_data_loader, setup_dataset

PRED_META = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)


def predict(df, process_fn, dataset, config):
    with torch.no_grad():
        # Note that `dataset` is a Hyrax DataProvider that is passed to `dist_data_loader`
        # It is not the bare dataset class.
        dataset.prepped_datasets["data"].data = df

        #! Be aware, this is Pytorch-ignite.
        data_loader = dist_data_loader(dataset, config)

        # NOTE: This is a placeholder, but we need to be more flexible because values
        # returned from a model can be 1) a bare tensor or 2) a dictionary of tensors.
        z = np.full(len(df), np.nan, np.float32)
        current_idx = 0

        for batch in data_loader:
            # The first `None` parameter is a placeholder for the pytorch ignite engine
            model_out = process_fn(None, batch)
            if len(batch):
                batch_size = len(batch["object_id"])
                z[current_idx : current_idx + batch_size] = model_out.to("cpu").numpy()
                current_idx += batch_size

    df["pred"] = z
    return df


class TorchModelPlugin(WorkerPlugin):
    def __init__(self, name, config):
        self.name = name  # plugins are keyed by name on the worker
        self.config = config
        self.model = None
        self.dataset = None

    def setup(self, worker):
        model_cls = fetch_model_class(self.config)
        model = model_cls(self.config, None)

        #! Load the model weights using config value
        # load_model_weights(self.config, model, 'infer')

        #! Load the `prepare_inputs` function and attach it to the model
        # model.prepare_inputs = load_prepare_inputs(<input_directory>)

        # This is superfluous but could be used for diagnostics
        model.load_token = uuid.uuid4().int >> 65

        model.eval()
        self.model = _auto_model(model)

        # Build the function that is called that processes data
        self.process_fn = create_process_func("infer_batch", "mps", self.model, self.config)

        # setup the dataset class
        datasets = setup_dataset(self.config, shuffle=False)
        self.dataset = datasets["infer"]

    def teardown(self, worker):
        self.model = None


def predict_with_plugin(df, plugin_name):
    return predict(
        df,
        get_worker().plugins[plugin_name].process_fn,
        get_worker().plugins[plugin_name].dataset,
        get_worker().plugins[plugin_name].config,
    )

In [8]:
client.register_plugin(TorchModelPlugin("test-model", h.config))

# we want to combine the output of the model with the existing metadata
met = tess_filtered_nans._operation.meta.copy(deep=True)
met["pred"] = PRED_META["pred"]

tess_pred = tess_filtered_nans.map_partitions(predict_with_plugin, plugin_name="test-model", meta=met)

tess_pred.compute()

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

2026-10-02 12:38:27,199 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<torch.utils.data.da': 
	{'sampler': None, 'batch_size': 64, 'collate_fn': <bound method CollationMixin.collate of Name: data (primary dataset)
  Dataset class: DataframeDataset
  Data location: {'ticid': 'int64[pyarrow]', 'ra_obj': 'double[pyarrow]', 'dec_obj': 'double[pyarrow]', 'lightcurve': {'cadenceno': 'int32[pyarrow]', 'mom_centr1': 'double[pyarrow]', 'mom_centr1_err': 'float[pyarrow]', 'mom_centr2': 'double[pyarrow]', 'mom_centr2_err': 'float[pyarrow]', 'pdcsap_flux': 'float[pyarrow]', 'pdcsap_flux_err': 'float[pyarrow]', 'pos_corr1': 'float[pyarrow]', 'pos_corr2': 'float[pyarrow]', 'psf_centr1': 'double[pyarrow]', 'psf_centr1_err': 'float[pyarrow]', 'psf_centr2': 'double[pyarrow]', 'psf_centr2_err': 'float[pyarrow]', 'quality': 'int32[pyarrow]', 'sap_bkg': 'float[pyarrow]', 'sap_bkg_err': 'float[pyarrow]', 'sap_flux': 'float[pyarrow]', 'sap_flux_err': 'float[pyarrow]', 'ti

[179.96686096 179.88894017 179.73018435 179.60992084 179.72920864
 179.53729062 179.64125937 179.79722556 179.77600521 179.38349577
 179.49456987 179.55656324 179.60042726]
tensor([179.9669, 179.8889, 179.7302, 179.6099, 179.7292, 179.5373, 179.6413,
        179.7972, 179.7760, 179.3835, 179.4946, 179.5566, 179.6004],
       device='mps:0')
[179.63732719 179.56047962 179.56047962 179.53646584 179.3501016 ]
tensor([179.6373, 179.5605, 179.5605, 179.5365, 179.3501], device='mps:0')


Input data contains NaN values. This may mean your model output is all NaNs. Consider setting config['data_set']['nan_mode'] = 'quantile' or 'zero' or writing a to_tensor() function for your model. Search hyrax readthedocs for 'to_tensor' to get started.
2026-10-02 12:38:38,600 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<torch.utils.data.da': 
	{'sampler': None, 'batch_size': 64, 'collate_fn': <bound method CollationMixin.collate of Name: data (primary dataset)
  Dataset class: DataframeDataset
  Data location: {'ticid': 'int64[pyarrow]', 'ra_obj': 'double[pyarrow]', 'dec_obj': 'double[pyarrow]', 'lightcurve': {'cadenceno': 'int32[pyarrow]', 'mom_centr1': 'double[pyarrow]', 'mom_centr1_err': 'float[pyarrow]', 'mom_centr2': 'double[pyarrow]', 'mom_centr2_err': 'float[pyarrow]', 'pdcsap_flux': 'float[pyarrow]', 'pdcsap_flux_err': 'float[pyarrow]', 'pos_corr1': 'float[pyarrow]', 'pos_corr2': 'float[pyarrow]', 'psf_centr1': 'double[pyarrow]', 'psf_cen

[180.80637496 180.09284384 180.05950308]
tensor([180.8064, 180.0928, 180.0595], device='mps:0')
[180.23100339 179.59209421 179.48613458 180.11779148]
tensor([180.2310, 179.5921, 179.4861, 180.1178], device='mps:0')


ticid      ra_obj   dec_obj  \
_healpix_29                                            
1801390049747628151   35028418  180.231003 -0.731668   
1801407390937259735  902599667  179.592094 -0.728306   
1801413923461859294   35017228  179.486135 -0.667983   
1801437502909795554  289512775  180.117791 -0.103421   
1849437955645283823   35088390  180.806375 -0.354058   
1849477626264654295  289512755  180.092844 -0.039708   
1849478524753555753  376786323  180.059503  0.042249   
1897524140783988621   35019784  179.637327  0.038598   
1897526176329951224  337733952   179.56048   0.08292   
1897526176329951224  337733952   179.56048   0.08292   
1897527144708845140  337733971  179.536466  0.177815   
1897547248087973751   96197975  179.350102  0.113772   
1945557543028358483  289512725  179.966861  0.131074   
1945557924347894929  376758233   179.88894  0.164427   
1945603786772284903  262782272  179.730184  0.674598   
1945603972457643070  337750673  179.609921  0.642193   
1945605559467551911  262782294  179.729209  0.793713   
1945606287117976888  337734099  179.537291  0.747558   
1945607782731432936  337750607  179.641259  0.872143   
1945617641001766722  262782311  179.797226  0.871664   
1945619139859449367  262782313  179.776005  0.880915   
1945697768892195288  446677641  179.383496  0.780604   
1945700639041892336  337734119   179.49457  0.823025   
1945701286358226074  337734120  179.556563  0.832628   
1945701612772606460  337734133  179.600427  0.888519   

                                                            lightcurve  \
_healpix_29                                                              
1801390049747628151  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1801407390937259735  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1801413923461859294  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1801437502909795554  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1849437955645283823  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1849477626264654295  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1849478524753555753  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1897524140783988621  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1897526176329951224  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1897526176329951224  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1897527144708845140  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1897547248087973751  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1945557543028358483  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1945557924347894929  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1945603786772284903  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945603972457643070  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945605559467551911  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945606287117976888  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945607782731432936  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945617641001766722  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945619139859449367  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945697768892195288  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945700639041892336  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945701286358226074  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945701612772606460  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   

                           pred  
_healpix_29                      
1801390049747628151  180.231003  
1801407390937259735  179.592087  
1801413923461859294  179.486130  
1801437502909795554  180.117798  
1849437955645283823  180.806381  
1849477626264654295  180.092850  
1849478524753555753  180.059509  
1897524140783988621  179.637329  
1897526176329951224  179.560486  
1897526176329951224  179.560486  
1897527144708845140  179.536469  
1897547248087973751  179.350098  
1945557543028358483  179.966858  
194

In [9]:
client.close()
cluster.close()